# ICLR End-to-End Pipeline (Detailed Notebook)

This notebook mirrors `iclr.py`, but includes more detailed explanation and excerpts from the
`sim_unctrb_*` scripts to clarify *what each step does and why*.

## 0) Configuration
We centralize paths and parameters so the pipeline is reproducible. These mirror the defaults
used in the CLI sequence and `iclr.py`.

In [ ]:
from pathlib import Path

# Base output directory
BASE_OUT = Path("pyident_results")

# Subdirectories (match iclr.py defaults)
ENSEMBLE_DIR = BASE_OUT / "fresh_ensemble"
X0_DIR = BASE_OUT / "fresh_ABx0"
PBH_DIR = BASE_OUT / "fresh_nonidentifiable_ABx0"
BOXPLOT_DIR = PBH_DIR / "boxplots"

# Sweep parameters
SPARSITY_GRID = "0.0:0.1:1.0"
NDIM_GRID = "2:1:10"
SAMPLES_PER_CELL = 10000

# Density filter
DENSITY_MIN = 0.3
DENSITY_MAX = 0.7
DENSITY_SOURCE = "AB"

# x0 sampling for score plots
X0_SAMPLES = 10
MASK_PS = [0.25, 0.5, 0.75]
OUTLIER_TRIM = 0.05

# PBH selection + estimation
MASK_PS_PBH = [0.25, 0.5, 0.75, 1.0]
PBH_THRESHOLD = 1e-6
SEED = 12345
T = 100
DT = 1.0
U_SCALE = 3.0
DWELL = 1
ALGOS = "DMDc"  # or e.g. "SINDy,DMDc,MOESP,NODE"

## 1) Sweep systems + save matrices (`sim_regcomb_ctrb`)
**Goal:** build a grid of `(A,B)` systems across sparsity × state dimension,
compute controllability rank, and save matrices to disk for downstream filtering.

**Artifacts:**
- `scores_summary.csv` (per-cell controllable fraction)
- `systems.csv` (row per system with metadata)
- `systems_matrices.npz` (A,B matrices aligned by `system_index`)

In [ ]:
from pyident.experiments import sim_regcomb_ctrb

args = sim_regcomb_ctrb.build_parser().parse_args([
    "--axes", "sparsity,ndim",
    "--sparsity-grid", SPARSITY_GRID,
    "--ndim-grid", NDIM_GRID,
    "--samples", str(SAMPLES_PER_CELL),
    "--outdir", str(ENSEMBLE_DIR),
    "--save-matrices",
])
if args.m is None:
    args.m = int(args.n)

sim_regcomb_ctrb.run(args)

## 2) Filter uncontrollable systems by density (`filter_unctrb_dataset`)
**Goal:** keep only uncontrollable systems whose density is inside a target band.
This is used to focus later analysis on *uncontrollable but not-too-sparse* regimes.

**Artifacts:**
- `systems_unctrb_d0.3_0.7.csv`
- `systems_unctrb_d0.3_0.7.npz`

In [ ]:
from pyident.experiments import filter_unctrb_dataset

args = filter_unctrb_dataset.build_parser().parse_args([
    "--outdir", str(ENSEMBLE_DIR),
    "--density-min", str(DENSITY_MIN),
    "--density-max", str(DENSITY_MAX),
    "--density-source", DENSITY_SOURCE,
])
filter_unctrb_dataset.run(args)

## 3) Identifiability score distributions (`sim_unctrb_x0_boxplot`)
**Goal:** for each uncontrollable `(A,B)`, sample multiple `x0` in different modes
(sphere vs masked sphere) and compute identifiability scores:
- PBH structured margin
- left-eigenvector score (`mu_min`)

**Artifacts:**
- `identifiability_scores.csv`
- boxplots per score vs x0 sampling mode

**Relevant excerpt (from `sim_unctrb_x0_boxplot.py`):**
```python
scores["pbh"] = float(pbh_margin_structured(A, B, x0))
Xaug = np.concatenate([x0.reshape(-1, 1), B], axis=1)
mu_vals = left_eigvec_overlap(A, Xaug)
scores["mu"] = float(np.min(mu_vals)) if mu_vals.size else 0.0
```

In [ ]:
from pyident.experiments import sim_unctrb_x0_boxplot

filtered_csv = ENSEMBLE_DIR / f"systems_unctrb_d{DENSITY_MIN:g}_{DENSITY_MAX:g}.csv"
filtered_npz = ENSEMBLE_DIR / f"systems_unctrb_d{DENSITY_MIN:g}_{DENSITY_MAX:g}.npz"

args = sim_unctrb_x0_boxplot.build_parser().parse_args([
    "--dataset-csv", str(filtered_csv),
    "--dataset-npz", str(filtered_npz),
    "--x0-samples", str(X0_SAMPLES),
    "--mask-ps", *[str(p) for p in MASK_PS],
    "--mask-renorm",
    "--outdir", str(X0_DIR),
    "--outlier-trim", str(OUTLIER_TRIM),
])

sim_unctrb_x0_boxplot.run(args)

## 4) Select low‑PBH triples + estimate (`sim_unctrb_pbh_estimators`)
**Goal:** re-sample `x0` (same policy), retain only triples with PBH below a threshold,
then simulate trajectories and run estimators on them.

**Artifacts:**
- `selected_pbh_lt_1e-06.csv/.npz` (the filtered `(A,B,x0)` triples)
- `estimation_errors.csv` (per‑algorithm errors)

**Relevant excerpt (from `sim_unctrb_pbh_estimators.py`):**
```python
pbh = pbh_margin_structured(A, B, x0)
if pbh < threshold:
    ...
    U = prbs(T, m, scale=u_scale, dwell=dwell, rng=rng)
    X = simulate_dt(x0, A, B, U)
    X0, X1 = X[:, :-1], X[:, 1:]
    ...
    Ahat, Bhat = estimator(X0, X1, U.T, dt)
```

In [ ]:
from pyident.experiments import sim_unctrb_pbh_estimators

args = sim_unctrb_pbh_estimators.build_parser().parse_args([
    "--dataset-csv", str(filtered_csv),
    "--dataset-npz", str(filtered_npz),
    "--outdir", str(PBH_DIR),
    "--seed", str(SEED),
    "--x0-samples", str(X0_SAMPLES),
    "--mask-ps", *[str(p) for p in MASK_PS_PBH],
    "--mask-renorm",
    "--pbh-threshold", str(PBH_THRESHOLD),
    "--T", str(T),
    "--dt", str(DT),
    "--u-scale", str(U_SCALE),
    "--dwell", str(DWELL),
    "--algos", ALGOS,
])

sim_unctrb_pbh_estimators.run(args)

## 5) Error boxplots (`sim_unctrb_pbh_error_boxplots`)
**Goal:** load the *exact* selected triples and re-run estimators to compute error
boxplots in both standard basis and `P`‑basis (visible subspace basis).

**Artifacts:**
- `boxplot_err_standard.png`
- `boxplot_err_Pbasis.png`
- `boxplot_err_standard_vs_Pbasis.png`

**Relevant excerpt (from `sim_unctrb_pbh_error_boxplots.py`):**
```python
Vbasis = build_visible_basis_dt(A, B, x0, tol=visible_tol)
P = orthonormal_completion(Vbasis)
A_P = P.T @ A @ P
B_P = P.T @ B
```

In [ ]:
from pyident.experiments import sim_unctrb_pbh_error_boxplots

selected_suffix = f"pbh_lt_{PBH_THRESHOLD:g}"
selected_csv = PBH_DIR / f"selected_{selected_suffix}.csv"
selected_npz = PBH_DIR / f"selected_{selected_suffix}.npz"

args = sim_unctrb_pbh_error_boxplots.build_parser().parse_args([
    "--selected-npz", str(selected_npz),
    "--selected-csv", str(selected_csv),
    "--outdir", str(BOXPLOT_DIR),
    "--seed", str(SEED),
    "--T", str(T),
    "--dt", str(DT),
    "--u-scale", str(U_SCALE),
    "--dwell", str(DWELL),
    "--algos", ALGOS,
])

sim_unctrb_pbh_error_boxplots.run(args)

---
## Notes
- This notebook runs the pipeline in-process (no shell commands).
- All artifacts match the CLI sequence and `iclr.py`.
- You can switch algorithms by setting `ALGOS` (e.g., `"SINDy,DMDc,MOESP,NODE"`).